# Four-Player Briscola RL Analysis

This notebook is a lightweight starting point for inspecting the environment, running a tiny self-play update, and evaluating the learned policy against simple baselines.

The full experiments should use larger batches and more seeds than the examples below.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import random

from briscola.baselines import RandomPolicy, GreedyTrickPolicy, SimpleHeuristicPolicy
from briscola.env import FourPlayerBriscolaEnv
from briscola.evaluation import evaluate_team_match
from briscola.features import BriscolaFeatureExtractor
from briscola.policies import LinearSoftmaxPolicy
from briscola.self_play import SelfPlayConfig, SelfPlayTrainer, SnapshotPool

## Inspect One Initial State

In [ ]:
env = FourPlayerBriscolaEnv(seed=42)
obs = env.observe(env.current_player)

print('Current player:', obs.player_id)
print('Partner:', obs.partner_id)
print('Trump suit:', obs.trump_suit)
print('Revealed trump:', obs.revealed_trump)
print('Hand:', [card.id for card in obs.hand])
print('Deck count:', obs.deck_count)

## Run A Random Match

In [ ]:
env = FourPlayerBriscolaEnv(seed=7)
policy = RandomPolicy()
rng = random.Random(7)

while not env.done:
    obs = env.observe(env.current_player)
    action = policy.select_action(obs, rng)
    env.step(action)

print('Final scores:', env.scores)
print('Tricks:', env.trick_index)
print('Cards per player:', env.cards_played_by_player())

## Tiny Self-Play Training Demo

This cell intentionally uses very small numbers so it runs quickly. Real experiments should use the values in `configs/train_pool_selfplay.yaml`.

In [ ]:
extractor = BriscolaFeatureExtractor()
learner = LinearSoftmaxPolicy.initialize(extractor, rng=random.Random(0), name='learner')
pool = SnapshotPool(feature_extractor=extractor, max_size=10)

config = SelfPlayConfig(
    batch_size=20,
    learning_rate=0.005,
    snapshot_interval=2,
)
trainer = SelfPlayTrainer(learner=learner, pool=pool, config=config, seed=123)

history = []
for update in range(5):
    stats = trainer.train_update()
    history.append(stats)
    print(update + 1, stats)

print('Pool size:', len(pool.snapshots))

## Evaluate Against Baselines

In [ ]:
seeds = list(range(100, 120))

for baseline in [RandomPolicy(), GreedyTrickPolicy(), SimpleHeuristicPolicy()]:
    result = evaluate_team_match(learner, baseline, seeds=seeds, paired=True, greedy=True)
    print('\nLearner vs', baseline.name)
    print(result)

## Inspect Learned Feature Weights

In [ ]:
weights = list(zip(extractor.feature_names, learner.theta))
for name, value in sorted(weights, key=lambda item: abs(item[1]), reverse=True)[:15]:
    print(f'{name:40s} {value:+.4f}')